In [ ]:
import pandas as pd
import os
import google.generativeai as genai
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) 
import panel as pn  # GUI

pn.extension()

In [ ]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=api_key, transport="rest")

In [ ]:
menu_df = pd.read_csv('menu.csv')
print(menu_df)
# print(menu_df.columns.tolist())
# print(menu_df.head())

In [ ]:
def format_menu_for_prompt(df):
    lines = []
    for category in df['category'].unique():
        lines.append(f"\n{category.upper()}S:")
        subset = df[df['category'] == category]
        for item in subset['item'].unique():
            item_rows = subset[subset['item'] == item]
            if item_rows['size'].notna().all() and (item_rows['size'] != '').all():
                size_prices = ", ".join(
                    f"{row['size']} ${row['price']:.2f}" for _, row in item_rows.iterrows()
                )
                lines.append(f"- {item}: {size_prices}")
            else:
                price = item_rows.iloc[0]['price']
                lines.append(f"- {item}: ${price:.2f}")
    return "\n".join(lines)

menu_text = format_menu_for_prompt(menu_df)
print(menu_text)

In [ ]:
def get_completion_from_messages(messages, model="gemini-3.8-flash", temperature=0):
    system_msg = None
    history = []

    for m in messages:
        if m['role'] == 'system':
            system_msg = m['content']
        elif m['role'] == 'user':
            history.append({'role': 'user', 'parts': [m['content']]})
        elif m['role'] == 'assistant':
            history.append({'role': 'model', 'parts': [m['content']]})

    gen_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_msg,
        generation_config=genai.types.GenerationConfig(temperature=temperature)
    )

    if history and history[-1]['role'] == 'user':
        last_message = history[-1]['parts'][0]
        chat_history = history[:-1]
    else:
        last_message = ""
        chat_history = history

    chat = gen_model.start_chat(history=chat_history)
    response = chat.send_message(last_message)

    return response.text

In [ ]:
def collect_messages(_):
    prompt = inp.value_input
    inp.value = ''

    if not prompt:
        return pn.Column(*panels)

    context.append({'role':'user', 'content':f"{prompt}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600,  styles={'background-color': '#E8F0FE', 'color': '#000000', 'padding': '8px', 'border-radius': '8px'})))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600,styles={'background-color':'#F6F6F6', 'color': '#000000', 'padding': '8px', 'border-radius': '8px'})))
    return pn.Column(*panels)

In [ ]:
panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes:
{menu_text}
"""} ]  # accumulate messages

inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Chat!")

interactive_conversation = pn.bind(collect_messages, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
    
)

dashboard   

In [ ]:
messages = context.copy()
messages.append(
    {'role':'user', 'content': "create a json summary of the previous food order. Itemize the price for each item. \
The fields should be 1) pizza, price 2) list of toppings, price 3) list of drinks, include size and price 4) list of sides, include size and price 5) total price"}
)

response = get_completion_from_messages(messages, temperature=0)
print(response)
